In [1]:
import subprocess
import os
import json
from pathlib import Path
import pdfplumber

In [ ]:
import psycopg2
from psycopg2 import sql, extras
import logging
import csv
import os
import json


In [2]:
def extract_to_standard_json(source_dir, target_dir, soffice_path):
    source_root = Path(source_dir).resolve()
    target_root = Path(target_dir).resolve()
    
    # Standardize output folders
    pdf_output_dir = target_root / "generated_pdfs"
    json_output_dir = target_root / "standardized_json"
    
    for folder in [pdf_output_dir, json_output_dir]:
        folder.mkdir(parents=True, exist_ok=True)
    
    clean_env = os.environ.copy()
    clean_env.pop("PYTHONHOME", None)
    clean_env.pop("PYTHONPATH", None)

    for file_path in source_root.rglob("*"):
        if file_path.is_dir() or file_path.suffix in [".pdf", ".json"] or file_path.name.startswith(".~lock"):
            continue

        print(f"Processing: {file_path.name}")
        
        try:
            # 1. Convert using the EXPLICIT Draw filter
            # Using 'pdf:draw_pdf_Export' ensures LibreOffice Draw handles the output
            subprocess.run([
                soffice_path, "--headless", 
                "--convert-to", "pdf:draw_pdf_Export",
                "--outdir", str(pdf_output_dir),
                str(file_path)
            ], check=True, env=clean_env, capture_output=True)
            
            # 2. Locate the resulting PDF
            # LibreOffice output name logic: OriginalName.pdf
            pdf_path = pdf_output_dir / f"{file_path.name}.pdf"
            json_path = json_output_dir / f"{file_path.name}.json"

            if not pdf_path.exists():
                print(f"   !! PDF generation failed for {file_path.name}. Check permissions or LibreOffice version.")
                continue

            # 3. Extraction logic
            json_data = {
                "source_file": str(file_path).replace("\\", "/"),
                "document_name": file_path.name,
                "page_count": 0,
                "metadata": {"title": "", "source": "QuarkXPress Legacy"},
                "pages": []
            }

            with pdfplumber.open(pdf_path) as pdf:
                json_data["page_count"] = len(pdf.pages)
                for i, page in enumerate(pdf.pages):
                    page_text = page.extract_text() or ""
                    json_data["pages"].append({
                        "page_name": str(i + 1),
                        "frames": [{
                            "bounds": [0, 0, 0, 0],
                            "paragraphs": [{"style": "Normal", "text": line} for line in page_text.split('\n')],
                            "text": page_text
                        }]
                    })
            
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(json_data, f, indent=4, ensure_ascii=False)
                
            print(f"   Done -> {json_path.name}")

        except Exception as e:
            print(f"   Failed {file_path.name}: {e}")

In [4]:
if __name__ == "__main__":

    INPUT_FOLDER = r"E:/Callproject/assembled/1_05-02 Call/Classifieds/Child Care"
    OUTPUT_FOLDER = r"E:/Callproject/c_out_quark"
    LIBREOFFICE_BIN = r"C:\Program Files\LibreOffice\program\soffice.exe" 

    extract_to_standard_json(INPUT_FOLDER, OUTPUT_FOLDER, LIBREOFFICE_BIN)


Processing: C0000-Licensed Home
   Done -> C0000-Licensed Home.json
Processing: C0520-Child Care avail
   Done -> C0520-Child Care avail.json
